In [1]:
import joblib
import os
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf

print("--- SENTRi-X: Cross-Dataset Validation (BoT-IoT) ---")

# Paths
processed_dir = "../data/processed/"
models_dir = "../models/"

# 1. Load the Hybrid "Brains"
print("Loading SENTRi-X Hybrid Models...")
rf_model = joblib.load(os.path.join(models_dir, "rf_model_ton_iot.joblib"))
cnn_model = tf.keras.models.load_model(os.path.join(models_dir, "cnn_model_ton_iot.h5"))

# 2. Load the BoT-IoT "Final Exam" Data
print("Loading Aligned BoT-IoT Tensors...")
X_bot = joblib.load(os.path.join(processed_dir, "bot_iot_X_test.pkl"))
y_bot = joblib.load(os.path.join(processed_dir, "bot_iot_y_test.pkl"))

print(f"Ready to evaluate {X_bot.shape[0]:,} unseen BoT-IoT packets.")

--- SENTRi-X: Cross-Dataset Validation (BoT-IoT) ---
Loading SENTRi-X Hybrid Models...


Loading Aligned BoT-IoT Tensors...
Ready to evaluate 144,728 unseen BoT-IoT packets.


In [2]:
print("Executing Hybrid Inference (Soft-Voting)...")

# Get probabilities from Random Forest
rf_probs = rf_model.predict_proba(X_bot)

# Get probabilities from CNN (reshape for the Conv1D layer)
X_bot_cnn = X_bot.reshape(X_bot.shape[0], X_bot.shape[1], 1)
cnn_probs = cnn_model.predict(X_bot_cnn)

# Average the probabilities (The SENTRi-X Fusion)
hybrid_probs = (rf_probs + cnn_probs) / 2
hybrid_preds = np.argmax(hybrid_probs, axis=1)

print("Inference Complete.")

Executing Hybrid Inference (Soft-Voting)...


c:\Users\LENOVO\SENTRi-X\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


4523/4523 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step
Inference Complete.


In [3]:
print("\n--- SENTRi-X Performance on BoT-IoT (Cross-Validation) ---")
print(f"Overall Accuracy: {accuracy_score(y_bot, hybrid_preds):.4%}")
print("\nDetailed Classification Report:")
print(classification_report(y_bot, hybrid_preds, target_names=['Normal', 'Attack']))

# Confusion Matrix for your thesis diagrams
cm = confusion_matrix(y_bot, hybrid_preds)
print("\nConfusion Matrix:")
print(cm)


--- SENTRi-X Performance on BoT-IoT (Cross-Validation) ---
Overall Accuracy: 0.0180%

Detailed Classification Report:
              precision    recall  f1-score   support

      Normal       0.00      1.00      0.00        26
      Attack       0.00      0.00      0.00    144702

    accuracy                           0.00    144728
   macro avg       0.00      0.50      0.00    144728
weighted avg       0.00      0.00      0.00    144728


Confusion Matrix:
[[    26      0]
 [144702      0]]


c:\Users\LENOVO\SENTRi-X\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\LENOVO\SENTRi-X\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\LENOVO\SENTRi-X\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [4]:
import joblib
import os
import pandas as pd

# 1. Load the special diagnostic DataFrame we saved in the ETL pipeline
processed_dir = "../data/processed/"
# This file contains the 'words' before they were turned into scaled numbers
X_bot_df = joblib.load(os.path.join(processed_dir, "bot_iot_X_test_df.pkl"))

print("--- SENTRi-X Diagnostic: Feature Alignment Check ---")

# 2. Check the Connection State columns (Are they all 0.0?)
conn_cols = [col for col in X_bot_df.columns if 'conn_state' in col]
print(f"Number of Connection State columns found: {len(conn_cols)}")
print("\nTop Connection State sums (Should NOT be 0.0):")
print(X_bot_df[conn_cols].sum().sort_values(ascending=False).head(5))

# 3. Check the Protocol columns
proto_cols = [col for col in X_bot_df.columns if 'proto' in col]
print("\nTop Protocol sums (Should NOT be 0.0):")
print(X_bot_df[proto_cols].sum().sort_values(ascending=False).head(5))

# 4. Check the Numerical Scaling ranges
# If these are massive (e.g., millions), the math units are wrong
print("\nNumerical Feature Ranges (Min/Max):")
print(X_bot_df[['src_bytes', 'duration', 'src_pkts']].describe().loc[['min', 'max']])

--- SENTRi-X Diagnostic: Feature Alignment Check ---
Number of Connection State columns found: 12

Top Connection State sums (Should NOT be 0.0):
conn_state_RSTR      35005
conn_state_S0        34646
conn_state_SF          369
conn_state_REJ           0
conn_state_RSTOS0        0
dtype: int64

Top Protocol sums (Should NOT be 0.0):
proto_udp    74329
proto_tcp    70103
dtype: int64

Numerical Feature Ranges (Min/Max):
     src_bytes  duration  src_pkts
min        0.0       0.0       0.0
max        0.0       0.0       0.0
